## Neural network: formal definition

A feedforward neural network with $L$ layers is a function $f: \mathbb{R}^{n_0} \to \mathbb{R}^{n_L}$ defined by the recurrence:

$$
\mathbf{h}_0 = \mathbf{x}
$$

$$
\mathbf{h}_l = \sigma_l(W_l \, \mathbf{h}_{l-1} + \mathbf{b}_l) \qquad l = 1, \dots, L
$$

$$
f(\mathbf{x}) = \mathbf{h}_L
$$

Where:

- $\mathbf{x} \in \mathbb{R}^{n_0}$ is the input vector
- $W_l \in \mathbb{R}^{n_l \times n_{l-1}}$ is the weight matrix of layer $l$
- $\mathbf{b}_l \in \mathbb{R}^{n_l}$ is the bias vector of layer $l$
- $\sigma_l$ is the activation function applied elementwise (ReLU, tanh, etc.)
- $n_l$ is the number of neurons in layer $l$
- The **architecture** is the choice of $(n_0, n_1, \dots, n_L)$ and each $\sigma_l$

The parameters of the network are $\theta = \{W_1, \mathbf{b}_1, \dots, W_L, \mathbf{b}_L\}$, so the network is written as $f(\mathbf{x}; \theta)$.

> **Note:** This formalization assumes the input is a flat vector $\mathbf{x} \in \mathbb{R}^{n_0}$. When the input has spatial or temporal structure (e.g. images, audio, video), it can either be flattened into a vector (losing structural information), or processed by architectures that operate on higher-dimensional tensors directly (e.g. CNNs for 2D spatial data, RNNs for sequences). Those architectures replace the matrix multiply $W_l \mathbf{h}_{l-1}$ with different linear operations (convolutions, recurrent steps) and require a different formalization, but the core principle — learnable parameters, nonlinearities, and optimization via gradient descent — remains the same.

## Training

Given a dataset $\{(\mathbf{x}_i, y_i)\}_{i=1}^N$, training finds:

$$
\theta^* = \arg\min_\theta \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}\big(f(\mathbf{x}_i; \theta),\; y_i\big)
$$

Where:

- $y_i \in \mathbb{R}^{n_L}$ is the target (same space as the network output)
- $\mathcal{L}: \mathbb{R}^{n_L} \times \mathbb{R}^{n_L} \to \mathbb{R}$ is the loss function (always scalar-valued)
- The loss must be scalar so that $\nabla_\theta \mathcal{L}$ yields a single direction in parameter space

This is solved iteratively via gradient descent:

$$
\theta \leftarrow \theta - \eta \, \nabla_\theta J(\theta)
$$

where $J(\theta) = \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}\big(f(\mathbf{x}_i; \theta), y_i\big)$ and $\eta$ is the learning rate. The gradients $\nabla_\theta J$ are computed efficiently via **backpropagation** (the chain rule applied layer by layer).

Es decir, supongamos que tenemos un input de $\mathbf{x} \in \mathbb{R}^{3}$. Y hemos decidido que nuestra primera capa va a tener 8 neuronas (decisión arbitraria). Entonces, la primera matriz de pesos será $W \in \mathbb{R}^{8 \text{neuronas} \times 3}$. Cada neurona tendrá un weight vector $\mathbf{w}_i \in \mathbb{R}^3$, y un solo bias para cada neurona. Por ello necesitamos $8 \times 3 = 24$ weights en total y solo $8$ biases.

Dada la definición formal arriba, es importante notar que el resultado $\mathbf{h}_l$ de la capa número $l$ será de dimensiones tal que $\mathbf{h}_l \in \mathbb{R}^{n_l}$, donde $n_l$ es el número de neuronas que esa capa $l$ tenía.

Ejemplo: del resultado anterior tendremos

$$
\underbrace{\mathbf{h}}_{8 \times 1} = \sigma\!\Big(
  \underbrace{W}_{8 \times 3}\;
  \underbrace{\mathbf{x}}_{3 \times 1} +
  \underbrace{\mathbf{b}}_{8 \times 1}
\Big)
$$

## Column or row vectors?

The above uses the **column vector** convention, i.e. we represent $\mathbf{x}$ by a column vector. Sometimes in `PyTorch` we use the **row vector** convention, for which we have to multiply `h = x @ W + b` instead of `h = W @ x + b`. In the row vector convention, we would have to transpose the matrix. It's equivalent.

## Softmax

### Intuition

The softmax function doesn't *make* the outputs "correct" probabilities. Softmax is just a shape-fixer: it takes any vector of real numbers and produces *a* valid probability distribution. That's a purely mechanical operation. It guarantees the output is *a* probability distribution, not that it's the *right* one.

If you initialize your network with random weights and run a forward pass, softmax will dutifully produce a probability distribution at the end — and it will be garbage. Random numbers in, valid-but-meaningless distribution out. So where does the "right"-ness come from?

**Training** is what aligns the distribution with reality. Here's the actual logic:

1. We *declare*, by architectural choice, that the network's output will be interpreted as a probability distribution. This is a modeling decision, not a discovery. We're saying: "I'm going to build a machine whose output I will read as $P(\text{next char} \mid \text{context})$."
2. For that declaration to make sense, the output must at least *look* like a probability distribution (non-negative, sums to 1). Softmax enforces that shape.
3. Then we train the network to make this declared distribution match the empirical distribution of the data. The loss function (negative log-likelihood / cross-entropy) literally measures "how badly does the distribution you're outputting explain the characters that actually appeared in the training set?" Gradient descent reduces that mismatch.

So the chain is:

$$\underbrace{\text{softmax}}_{\text{shape}} \;+\; \underbrace{\text{NLL loss}}_{\text{meaning}} \;+\; \underbrace{\text{gradient descent}}_{\text{fitting}} \;\;\Longrightarrow\;\; \text{outputs}\approx\text{real probabilities}$$

Softmax alone gives you something probability-*shaped*. The loss function is what gives those numbers their *semantics* — it punishes the network whenever the probability it assigns to the true next character is low, so the network is forced to push probability mass toward characters that actually occur in context. Over many gradient steps, the only way to reduce that loss is to actually learn the real conditional distribution.

### Formal definition

Given a vector of real numbers $z = (z_1, z_2, \ldots, z_n)$, softmax produces a new vector $p = (p_1, p_2, \ldots, p_n)$ where each entry is:

$$p_i = \frac{e^{z_i}}{\sum_{j=1}^{n} e^{z_j}}$$

#### A concrete toy example

Say your network outputs logits $z = (2.0,\ 1.0,\ 0.1)$ for three classes.

Step 1 — exponentiate each:

$$e^{2.0} \approx 7.39, \quad e^{1.0} \approx 2.72, \quad e^{0.1} \approx 1.11$$

Step 2 — sum them: $7.39 + 2.72 + 1.11 \approx 11.22$

Step 3 — divide each by the sum:

$$p_1 \approx \frac{7.39}{11.22} \approx 0.66, \quad p_2 \approx \frac{2.72}{11.22} \approx 0.24, \quad p_3 \approx \frac{1.11}{11.22} \approx 0.10$$

Check: $0.66 + 0.24 + 0.10 = 1.00$. ✓

So the logit vector $(2.0, 1.0, 0.1)$ becomes the probability distribution $(0.66, 0.24, 0.10)$. The largest logit becomes the largest probability, as you'd expect.

#### Why the name "soft-max"

Compare to a "hard max": given $(2.0, 1.0, 0.1)$, a hard max would return $(1, 0, 0)$ — the winner takes everything, all other entries get zero. That's a hard, discontinuous decision.

Softmax is the smooth version. It still *emphasizes* the largest value (0.66 is quite a bit bigger than 0.24), but it assigns some probability to the others too. Crucially, it's differentiable everywhere — which is what you need for gradient descent to work. A hard max has no useful gradient; softmax has clean gradients everywhere.

And the "softness" is tunable: the more spread out the logits are, the more peaked the softmax output. Logits $(10, 1, 0.1)$ give almost all probability to the first class; logits $(0.1, 0.1, 0.1)$ give a uniform distribution. Large logit differences → confident prediction, small differences → uncertain prediction.

#### Two practical details worth knowing

*Shift invariance*: adding the same constant to every logit doesn't change the output. This is the answer to the check question I asked you earlier:

$$\frac{e^{z_i + c}}{\sum_j e^{z_j + c}} = \frac{e^c \cdot e^{z_i}}{e^c \cdot \sum_j e^{z_j}} = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

The $e^c$ factors cancel. This means only the *differences* between logits matter, not their absolute values. It's also why, in practice, implementations subtract $\max(z)$ from all logits before exponentiating — it prevents numerical overflow without changing the answer.

*The output always has full support*: every $p_i > 0$, strictly. Softmax never assigns exactly zero probability to anything, because $e^x$ is never zero. In a character model this means even absurd continuations get some tiny probability — which is actually a feature, not a bug, because zero probabilities would blow up the log in the loss function.

### Logits

Softmax takes logits $z$ and produces probabilities $p$:

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Now ask: given the probabilities $p$, what were the logits? Solve for $z_i$. Take the log of both sides:

$$\log p_i = z_i - \log \sum_j e^{z_j}$$

That second term on the right is the same for every $i$ — it's just some constant $C$ that doesn't depend on which entry you're looking at. So:

$$z_i = \log p_i + C$$

A logit is **the log of a probability** (plus an irrelevant constant that softmax would erase anyway). The name is just shorthand for "log-of-probability-thing."

#### Origin

The term was coined in 1944 by the statistician Joseph Berkson for binary logistic regression, where you model a single probability $p$ with:

$$\text{logit}(p) = \log \frac{p}{1 - p}$$

That's the "log-odds" — the log of the ratio of the probability of success to the probability of failure. Berkson called it "logit" as a deliberate play on "probit" (another competing model at the time, short for "probability unit"). So "logit" literally meant "logistic unit" — the natural input to the logistic/sigmoid function